# Module 01 - RAG Concepts

**Duration:** 60 minutes

In this module we look at what RAG is, why it exists, and how the pieces fit together.
Then we build a minimal version from scratch using only numpy, so every step is visible
before we start using libraries to handle it for us.

---


## 1.1 Why RAG?

Large language models are trained on a fixed snapshot of text. Once training is done,
their knowledge is frozen. This creates a few practical problems:

- They have no access to your private documents.
- They cannot answer questions about events after their training cutoff.
- Retraining a model to update its knowledge is expensive and slow.

Retrieval Augmented Generation (RAG) sidesteps all of this. Instead of trying to bake
knowledge into the model weights, we retrieve the relevant information at query time
and hand it to the model as part of the prompt. The model does not need to remember
anything - it just needs to read and reason.

This makes RAG practical for several use cases that a plain LLM cannot handle well:

- Querying internal company documents, wikis, or support tickets.
- Answering questions about content that changes frequently (pricing, policies, news).
- Personalised responses grounded in a specific user's data.
- Reducing hallucinations by giving the model a source to read from.


## 1.2 How it works

The full pipeline has two phases.

**Indexing phase** (done once, or when documents change):

```
Documents
   |
   v
Chunker      splits text into overlapping pieces
   |
   v
Embedder     converts each chunk into a vector
   |
   v
Vector DB    stores and indexes the vectors
```

**Query phase** (done on every user question):

```
User question
   |
   v
Embedder     same model, same vector space
   |
   v
Vector DB    finds the most similar chunks
   |
   v
Prompt       question + retrieved context
   |
   v
LLM          generates the answer
```

The key insight is that the retrieval step happens in vector space.
Text gets turned into numbers, and similarity between texts becomes
a geometric distance between points. We will look closely at how
this works in Module 02.


## 1.3 Limitations to keep in mind

RAG is not a magic fix. The three stages each have their own failure modes.

**Indexing problems.** If you chunk documents badly, you will lose context at the
boundaries. If your documents are noisy or poorly written, the embeddings will
be noisy too. Garbage in, garbage out.

**Retrieval problems.** The retriever might not return the chunk that actually
answers the question. This can happen when the question uses different vocabulary
than the document, or when the relevant information is spread across multiple chunks.

**Generation problems.** Even with good context, the LLM can still produce wrong
answers. It might ignore the context and use its parametric knowledge instead,
or it might misread the retrieved text. Modules 05 and 06 deal with detecting
and reducing these problems.


---

## 1.4 RAG from scratch

Before using any libraries, we build a minimal RAG system using only numpy.
It will not scale, but it makes every step explicit.

We need:
1. A way to encode text as vectors.
2. A way to find the most similar vector to a query.
3. A way to call an LLM with the retrieved context.


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

EMBEDDING_MODEL = 'all-MiniLM-L6-v2'
model = SentenceTransformer(EMBEDDING_MODEL)
print('Model loaded.')


In [ ]:
# Our tiny document corpus
documents = [
    "Bali's beautiful beaches and rich culture make it a popular travel destination.",
    "Pizza in Rome is famous for its thin crust, fresh ingredients, and wood-fired ovens.",
    "Graphics processing units have become essential for training AI models.",
    "Newton's laws of motion transformed our understanding of physics.",
    "The French Revolution played a crucial role in shaping contemporary France.",
    "Maintaining good health requires regular exercise, a balanced diet, and quality sleep.",
    "Global warming threatens ecosystems and wildlife across the planet.",
    "The AI Service Center Berlin-Brandenburg offers workshops, consulting, and compute resources.",
    "Django Reinhardt's jazz compositions are celebrated for their captivating melodies.",
]

# Encode all documents into vectors
doc_embeddings = model.encode(documents)

print('Shape of embedding matrix:', doc_embeddings.shape)
print('Each document becomes a vector of', doc_embeddings.shape[1], 'numbers.')


Each document is now a point in a 384-dimensional space.
Documents that talk about similar things should be nearby in this space.
That is the core idea behind semantic search.


In [ ]:
# Cosine similarity measures the angle between two vectors.
# Two vectors pointing in the same direction have similarity 1.
# Two perpendicular vectors have similarity 0.

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def retrieve(query: str, top_k: int = 3) -> list[tuple[str, float]]:
    query_embedding = model.encode(query)
    scores = [cosine_similarity(query_embedding, doc_emb) for doc_emb in doc_embeddings]
    ranked = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]


query = 'I want to learn about artificial intelligence in Berlin'
results = retrieve(query)

print(f'Query: {query}\n')
for doc, score in results:
    print(f'  score {score:.3f}: {doc}')


In [ ]:
# Now add the LLM step
import requests
import json
from os import getenv
from urllib.parse import urljoin

OLLAMA_URL = urljoin(getenv('OLLAMA_HOST', 'http://localhost:11434'), 'api')
MODEL = 'llama3.2'


def generate(prompt: str, temp: float = 0.3) -> str:
    url = OLLAMA_URL + '/generate'
    data = {'model': MODEL, 'prompt': prompt, 'stream': False,
            'options': {'temperature': temp}}
    r = requests.post(url, json=data)
    return json.loads(r.text).get('response', '')


def rag(query: str, top_k: int = 3) -> str:
    # Step 1: retrieve
    results = retrieve(query, top_k=top_k)
    context = '\n'.join(doc for doc, _ in results)

    # Step 2: build prompt
    prompt = (
        'Use the following context to answer the question. '
        'Keep the answer concise.\n\n'
        f'Context:\n{context}\n\n'
        f'Question: {query}\n'
        'Answer:'
    )

    # Step 3: generate
    return generate(prompt)


answer = rag('What AI resources are available in Berlin?')
print(answer)


That is the full pipeline: encode, retrieve, prompt, generate.
Everything in the rest of the workshop is an improvement on one of those four steps.

---

**Exercises**

1. Change the query to something unrelated to the documents (e.g. 'Who invented the telephone?').
   What does the retriever return? What does the LLM say?

2. Add three new documents to the `documents` list on a topic of your choice,
   re-encode, and ask a question about them.

3. Print the full prompt that gets sent to the LLM. Does reading it help explain the answer?

---

**Further reading**

- Original RAG paper: https://arxiv.org/abs/2005.11401
- Sentence Transformers: https://www.sbert.net/
